# Evaluate perception and execution

**Time:** 90–120 minutes  
**Prerequisites:** notebooks 00–03 and at least one completed or partially completed run.

## Learning objectives

- Separate model output, planned observations, evaluation data, and execution results
- Measure position, colour, and yaw performance instead of judging one attractive image
- Find the worst examples and turn them into a concrete next experiment
- Compare several runs without using ground truth to command the robot

> **Hard rule:** `ground_truth.json` and `perception_evaluation.json` are for scoring only. Never copy their coordinates into `detections.json` or `sort_plan.json`.

In [ ]:
import json
import statistics
from pathlib import Path
import matplotlib.pyplot as plt

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / 'fanuc_er4ia_sim' / 'run_sort_pipeline.sh').is_file():
            return p
    raise FileNotFoundError('Open this notebook from the hackathon repository')

ROOT = find_root()
SIM = ROOT / 'fanuc_er4ia_sim'
RUNS = SIM / 'runs'
candidates = sorted((p for p in RUNS.glob('*') if p.is_dir()), key=lambda p: p.stat().st_mtime, reverse=True) if RUNS.exists() else []
RUN = candidates[0] if candidates else RUNS / 'stack_01'
print('Selected run:', RUN)
print('Edit RUN above if you want a different experiment.')

## 1. Know the evidence

A normal run progresses through four kinds of evidence:

1. `detections.json`: parsed Cosmos output.
2. `sort_plan.json`: image-derived observations, gates, and executable picks.
3. `perception_evaluation.json`: one-to-one comparison against simulator truth, created only after the plan is saved.
4. `execution_report.json`: what PhysX says was actually picked and placed.

A missing later file tells you which stage failed. It does not mean the earlier stage was accurate.

In [ ]:
ARTIFACTS = ['world_camera.png', 'detections.json', 'sort_plan.json',
             'detections_overlay.png', 'perception_evaluation.json',
             'execution_report.json']
for name in ARTIFACTS:
    path = RUN / name
    print(f"{'OK':7} {name}" if path.exists() else f"MISSING {name}")

def read_json(name):
    path = RUN / name
    return json.loads(path.read_text()) if path.exists() else None

detections = read_json('detections.json')
plan = read_json('sort_plan.json')
evaluation = read_json('perception_evaluation.json')
report = read_json('execution_report.json')

## 2. Look before calculating

Inspect the raw capture and overlay side by side. Ask: Are all six cubes visible? Are labels attached to the correct objects? Are skipped objects red? Is the spread direction plausible?

In [ ]:
images = [(RUN / n, title) for n, title in [('world_camera.png', 'Raw capture'), ('detections_overlay.png', 'Planned observations')] if (RUN / n).exists()]
if images:
    fig, axes = plt.subplots(1, len(images), figsize=(8 * len(images), 6))
    axes = [axes] if len(images) == 1 else axes
    for ax, (path, title) in zip(axes, images):
        ax.imshow(plt.imread(path))
        ax.set_title(title)
        ax.axis('off')
    plt.show()
else:
    print('No images yet. Run notebook 03 or the full pipeline first.')

## 3. Calculate useful metrics

For the cube task, useful measures are colour accuracy, position error, yaw error modulo 90°, missed objects, unmatched detections, and whether all six observations passed every safety gate. Report the distribution and worst case—not only the average.

In [ ]:
if evaluation:
    matches = [m for m in evaluation.get('matches', []) if m.get('category') == 'cube']
    pos = [m['position_error_mm'] for m in matches if 'position_error_mm' in m]
    yaw = [m['yaw_error_deg_mod90'] for m in matches if 'yaw_error_deg_mod90' in m]
    colour_n = sum(bool(m.get('colour_correct')) for m in matches)
    print(f'Cubes matched: {len(matches)}/6')
    print(f'Colour accuracy: {colour_n}/{len(matches)}' if matches else 'Colour accuracy: n/a')
    if pos: print(f'Position error: median={statistics.median(pos):.2f} mm, worst={max(pos):.2f} mm')
    if yaw: print(f'Yaw error mod 90: median={statistics.median(yaw):.2f}°, worst={max(yaw):.2f}°')
    print('Missed:', evaluation.get('missed_objects', []))
    print('Unmatched:', evaluation.get('unmatched_detections', []))
else:
    print('No perception evaluation. The planner creates it when ground_truth.json is present.')

In [ ]:
if evaluation:
    cubes = [m for m in evaluation.get('matches', []) if m.get('category') == 'cube']
    names = [m.get('name', m.get('detection_id', '?')) for m in cubes]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(names, [m.get('position_error_mm', 0) for m in cubes])
    axes[0].set(title='Position error', ylabel='millimetres')
    axes[1].bar(names, [m.get('yaw_error_deg_mod90', 0) for m in cubes])
    axes[1].set(title='Yaw error modulo 90°', ylabel='degrees')
    for ax in axes: ax.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

## 4. Perception success is not task success

A correct label can still produce a failed pickup. Read the execution report and identify the first failed gate. The capstone requires `task_complete: true`, six verified pickups, and six verified placements.

In [ ]:
if report:
    print('task_complete:', report.get('task_complete'))
    print('verified_pickups:', report.get('verified_pickups'))
    print('verified_placements:', report.get('verified_placements'))
    print('abort_reason:', report.get('abort_reason'))
    for i, event in enumerate(report.get('picks', [])):
        if not event.get('placement_verified'):
            print('First failed event:', i, json.dumps(event, indent=2))
            break
else:
    print('No execution report. Isaac may not have run, or it may have stopped before writing one.')

## 5. Compare runs

Run the next cell after several experiments. Change one variable at a time and name run directories clearly. Do not compare a prompt change and a lighting change as if they were one experiment.

In [ ]:
def run_summary(run):
    def load(name):
        p = run / name
        return json.loads(p.read_text()) if p.exists() else {}
    ev, ex = load('perception_evaluation.json'), load('execution_report.json')
    cubes = [m for m in ev.get('matches', []) if m.get('category') == 'cube']
    pos = [m['position_error_mm'] for m in cubes if 'position_error_mm' in m]
    yaw = [m['yaw_error_deg_mod90'] for m in cubes if 'yaw_error_deg_mod90' in m]
    return {'run': run.name, 'matched': len(cubes), 'worst_mm': max(pos, default=None),
            'worst_yaw': max(yaw, default=None), 'pickups': ex.get('verified_pickups'),
            'placements': ex.get('verified_placements'), 'complete': ex.get('task_complete')}

rows = [run_summary(p) for p in sorted(RUNS.glob('*')) if p.is_dir()] if RUNS.exists() else []
for row in rows: print(row)

## Team exercise and checkpoint

1. Find the run's worst position or yaw error.
2. Inspect its raw image, overlay, raw Cosmos reply, and planned observation.
3. Write one hypothesis and one experiment that changes a single factor.
4. Predict the metric that should improve before running it.

You are finished when another teammate can answer: **what failed, what evidence supports that conclusion, and what single experiment comes next?** Do not lower a confidence or geometry gate just to turn the report green.